[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/00_setup_and_tests.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/aditya-siraskar/PINN_FTRE/blob/main/notebooks/00_setup_and_tests.ipynb)

# 00 · Setup and test suite

**Phase:** all

**Goal:** Check the environment, run the tests of the reference implementations, preview the figure style.

**Outputs:** test report; style preview

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive (Colab) or /kaggle/working (Kaggle), install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # Colab: keep figures/results across sessions (recommended). Ignored on Kaggle.
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
ON_KAGGLE = os.path.exists("/kaggle/working")   # needs Settings > Internet on, Accelerator > GPU T4 x2
if ON_KAGGLE:
    BASE = "/kaggle/working"   # kept with the notebook's output when run via Save Version
elif USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    rc = os.system(f'git -C "{ROOT}" pull -q --ff-only')
    if rc != 0:   # usually: result files on Drive that have since been committed to GitHub
        print("WARNING: git pull failed; the Drive copy is out of date. To sync (keeps untracked files), run:")
        print(f'!git -C "{ROOT}" fetch -q origin && git -C "{ROOT}" reset -q --hard origin/main')
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "none (Colab: Runtime > Change runtime type > T4 GPU; Kaggle: Settings > Accelerator > GPU T4 x2)")

## 1. Tests
All six must pass before any stage is trusted.

In [ ]:
!python -m pytest -q

## 2. Figure style preview
Each method keeps one color, linestyle and marker for the whole project.

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

In [ ]:
x = np.linspace(0, 1, 50)
fig, ax = plt.subplots(figsize=(7, 3.2))
for i, (k, s) in enumerate(P.METHOD.items()):
    ax.plot(x, x * 0 + i, **P.style(k), marker=s["marker"] or None, markevery=7)
ax.set(yticks=[], title="Method styles used in every figure", xlabel="")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.05))
plt.show()